# Capital Raise Detector

Predict the likelihood of companies performing primary equity raises based on 5 financial signals.

## Overview

This notebook analyzes companies on:
1. **Cash Runway** - How many months of cash remains
2. **Liquidity Stress** - Current/quick ratios and working capital
3. **Debt Maturity** - Near-term debt obligations
4. **Operational Red Flags** - Revenue trends, margins, FCF
5. **Market & Behavioral Signals** - Stock price, insider activity

Companies scoring > 50 are flagged for potential capital raise activity.

In [1]:
# Import required modules
from analyzer import CapitalRaiseAnalyzer
from data import EXAMPLE_COMPANIES, EARNINGS_CALL_TRANSCRIPT, MARKET_NEWS
from models import FinancialMetrics
import pandas as pd
from IPython.display import display, Markdown


## Initialize Analyzer

Create analyzer instance with your preferred LLM model.

In [2]:
# Initialize with default model (gpt-4o-mini)
# For higher accuracy, use 'gpt-4' or 'gpt-4o'
analyzer = CapitalRaiseAnalyzer(model_name="gpt-4o-mini")
print("✓ Analyzer initialized")

✓ Analyzer initialized


## Analyze Example Companies

Run the detector on our 5 example companies with different risk profiles.

In [24]:
# Analyze all example companies
predictions = analyzer.batch_analyze(EXAMPLE_COMPANIES)

# Display summary
summary_data = [
    {
        "Ticker": p.ticker,
        "Company": p.company_name,
        "Score": f"{p.likelihood_score:.1f}",
        "Risk Level": p.risk_level.upper(),
        "Alert": "⚠️  YES" if p.above_threshold else "✓ No",
        "Confidence": f"{p.confidence:.0f}%",
    }
    for p in predictions
]

summary_df = pd.DataFrame(summary_data)
display(Markdown("### Summary of All Companies"))
display(summary_df)

### Summary of All Companies

,Ticker,Company,Score,Risk Level,Alert,Confidence
0,STRONG,Strong Tech Inc,0.0,LOW,✓ No,85%
1,BURN,Burn Fast Corp,75.0,CRITICAL,⚠️ YES,85%
2,STRESS,Struggling Industries,97.0,CRITICAL,⚠️ YES,85%
3,GROW,Growth Ventures LLC,55.0,MEDIUM,⚠️ YES,85%
4,STABLE,Mature Corp Inc,0.0,LOW,✓ No,85%


## Detailed Analysis by Company

Inspect individual company scores and key drivers.

In [17]:
# Show detailed results for each company
for pred in predictions:
    print(pred)


Capital Raise Detector Report
Company: Strong Tech Inc (STRONG)
Likelihood Score: 0.0/100
Risk Level: LOW
Status: ✓ Below threshold
Confidence: 85.0%

Signal Breakdown:
  • Cash Runway: 0.0/40
  • Liquidity Stress: 0.0/20
  • Debt Maturity: 0.0/15
  • Operational Red Flags: 0.0/15
  • Market & Behavioral: 0.0/10

Key Drivers:



Capital Raise Detector Report
Company: Burn Fast Corp (BURN)
Likelihood Score: 75.0/100
Risk Level: CRITICAL
Status: ⚠️  ABOVE THRESHOLD
Confidence: 85.0%

Signal Breakdown:
  • Cash Runway: 40.0/40
  • Liquidity Stress: 0.0/20
  • Debt Maturity: 12.0/15
  • Operational Red Flags: 15.0/15
  • Market & Behavioral: 8.0/10

Key Drivers:
  - Low cash runway: 5.0 months
  - $20M debt due in 12 months
  - Negative operating cash flow
  - Elevated insider selling activity


Capital Raise Detector Report
Company: Struggling Industries (STRESS)
Likelihood Score: 97.0/100
Risk Level: CRITICAL
Status: ⚠️  ABOVE THRESHOLD
Confidence: 85.0%

Signal Breakdown:
  • Cash Runw

## Signal Breakdown

Visualize the contribution of each signal to the overall score.

In [18]:
# Create signal breakdown for each company
signal_data = []
for p in predictions:
    signal_data.append({
        "Ticker": p.ticker,
        "Cash Runway (0-40)": f"{p.signal_scores.cash_runway_score:.1f}",
        "Liquidity (0-20)": f"{p.signal_scores.liquidity_stress_score:.1f}",
        "Debt Maturity (0-15)": f"{p.signal_scores.debt_maturity_score:.1f}",
        "Operational (0-15)": f"{p.signal_scores.operational_red_flags_score:.1f}",
        "Market (0-10)": f"{p.signal_scores.market_behavioral_score:.1f}",
        "Total (0-100)": f"{p.likelihood_score:.1f}",
    })

signals_df = pd.DataFrame(signal_data)
display(Markdown("### Signal Score Breakdown"))
display(signals_df)

### Signal Score Breakdown

,Ticker,Cash Runway (0-40),Liquidity (0-20),Debt Maturity (0-15),Operational (0-15),Market (0-10),Total (0-100)
0,STRONG,0.0,0.0,0.0,0.0,0.0,0.0
1,BURN,40.0,0.0,12.0,15.0,8.0,75.0
2,STRESS,40.0,17.0,15.0,15.0,10.0,97.0
3,GROW,40.0,0.0,1.0,13.0,1.0,55.0
4,STABLE,0.0,0.0,0.0,0.0,0.0,0.0


## Alerts

Companies scoring above 50 (threshold) that need monitoring.

In [19]:
# Get companies above threshold
alerts = analyzer.get_alerts(predictions)

if alerts:
    display(Markdown(f"### ⚠️  {len(alerts)} Companies Above Threshold"))
    for alert in alerts:
        print(f"\n**{alert.ticker}** - Score: {alert.likelihood_score:.1f}")
        print(f"Risk Level: {alert.risk_level.upper()}")
        print("Key Drivers:")
        for driver in alert.key_drivers:
            print(f"  • {driver}")
else:
    print("No companies above threshold.")

### ⚠️  3 Companies Above Threshold


**STRESS** - Score: 97.0
Risk Level: CRITICAL
Key Drivers:
  • Low cash runway: 3.8 months
  • Weak liquidity: Current ratio 0.82
  • $50M debt due in 12 months
  • Negative operating cash flow
  • Revenue decline: -16.7%

**BURN** - Score: 75.0
Risk Level: CRITICAL
Key Drivers:
  • Low cash runway: 5.0 months
  • $20M debt due in 12 months
  • Negative operating cash flow
  • Elevated insider selling activity

**GROW** - Score: 55.0
Risk Level: MEDIUM
Key Drivers:
  • Low cash runway: 5.3 months
  • Negative operating cash flow


## With Market Signal Analysis

Include LLM analysis of earnings calls and news (requires OpenAI API key).

In [7]:
# Re-analyze with market signals
# This uses the LLM to analyze earnings call transcripts and news
# WARNING: This makes API calls to OpenAI

print("Analyzing company with market signals...")
enhanced_pred = analyzer.analyze(
    EXAMPLE_COMPANIES[1],  # BURN company
    earnings_call_transcript=EARNINGS_CALL_TRANSCRIPT,
    market_news=MARKET_NEWS,
)

print(enhanced_pred)
print(f"\nMarket signals added: {len(enhanced_pred.key_drivers)} key drivers identified")

Analyzing company with market signals...

Capital Raise Detector Report
Company: Burn Fast Corp (BURN)
Likelihood Score: 75.0/100
Risk Level: CRITICAL
Status: ⚠️  ABOVE THRESHOLD
Confidence: 85.0%

Signal Breakdown:
  • Cash Runway: 40.0/40
  • Liquidity Stress: 0.0/20
  • Debt Maturity: 12.0/15
  • Operational Red Flags: 15.0/15
  • Market & Behavioral: 8.0/10

Key Drivers:
  - Low cash runway: 5.0 months
  - $20M debt due in 12 months
  - Negative operating cash flow
  - Elevated insider selling activity


Market signals added: 4 key drivers identified


## Custom Company Analysis

Analyze your own company data.

In [9]:
# Example: Create custom company
from models import FinancialMetrics

my_company = FinancialMetrics(
    ticker="MYCORP",
    company_name="My Custom Company",
    cash_and_equivalents=100_000_000,
    monthly_burn_rate=5_000_000,
    current_assets=150_000_000,
    current_liabilities=80_000_000,
    quick_assets=120_000_000,
    accounts_payable=30_000_000,
    total_debt=50_000_000,
    debt_due_12mo=10_000_000,
    debt_due_6_18mo=15_000_000,
    revenue_trailing_12m=500_000_000,
    revenue_prior_year=450_000_000,
    operating_cash_flow_trailing_12m=50_000_000,
    gross_margin=0.60,
    prior_gross_margin=0.58,
    capex_annual=20_000_000,
    stock_price=100.0,
    stock_price_52w_high=110.0,
    insider_selling_activity="normal",
    auditor_going_concern=False,
    credit_rating="BBB",
    credit_rating_outlook="stable",
)

# Analyze
custom_result = analyzer.analyze(my_company)
print(custom_result)


Capital Raise Detector Report
Company: My Custom Company (MYCORP)
Likelihood Score: 0.0/100
Risk Level: LOW
Status: ✓ Below threshold
Confidence: 85.0%

Signal Breakdown:
  • Cash Runway: 0.0/40
  • Liquidity Stress: 0.0/20
  • Debt Maturity: 0.0/15
  • Operational Red Flags: 0.0/15
  • Market & Behavioral: 0.0/10

Key Drivers:




## Analyze by Ticker (Auto-fetch from SEC)

Enter any US public company ticker and automatically pull financial metrics from SEC EDGAR filings and Yahoo Finance.

In [11]:
from sec_fetcher import fetch_metrics

# Change this ticker to any US public company
TICKER = "GKOS"

# Fetch metrics automatically from SEC EDGAR + Yahoo Finance
metrics = fetch_metrics(TICKER)

# Run the analysis
result = analyzer.analyze(metrics)
print(result)

Looking up GKOS...
  Found: GLAUKOS Corp (CIK: 1192448)
  Fetching SEC filings...
  Fetching credit rating...
  Fetching stock data...
  Done! Metrics loaded for GLAUKOS Corp

Capital Raise Detector Report
Company: GLAUKOS Corp (GKOS)
Likelihood Score: 15.0/100
Risk Level: LOW
Status: ✓ Below threshold
Confidence: 85.0%

Signal Breakdown:
  • Cash Runway: 0.0/40
  • Liquidity Stress: 0.0/20
  • Debt Maturity: 0.0/15
  • Operational Red Flags: 15.0/15
  • Market & Behavioral: 0.0/10

Key Drivers:
  - Negative operating cash flow



## Screen All US Public Companies

Scan all ~10,000+ SEC filers and find companies flagged as high risk for capital raises. Uses the SEC EDGAR XBRL Frames API for efficient bulk data retrieval.

**Note:** This takes 1-2 minutes to fetch data from SEC EDGAR.

In [3]:
from screener import screen_all_companies

# Screen all US public companies
# Uses SEC EDGAR Frames API (bulk data, ~13 API calls total)
high_risk = screen_all_companies()

# Display results as a table
if high_risk:
    screen_data = [
        {
            "Ticker": ticker,
            "Company": name,
            "Score": f"{pred.likelihood_score:.1f}",
            "Risk Level": pred.risk_level.upper(),
            "Key Drivers": "; ".join(pred.key_drivers[:3]),
        }
        for ticker, name, pred in high_risk[:50]  # Show top 50
    ]

    screen_df = pd.DataFrame(screen_data)
    display(Markdown(f"### High-Risk Companies ({len(high_risk)} total, showing top 50)"))
    display(screen_df)
else:
    print("No high-risk companies found.")

Screening NYSE/Nasdaq companies (CY2025 Q1-Q4)...
  Minimum market cap: $1000M
Step 1: Fetching financial data from SEC EDGAR...
  Fetching balance sheet data (Q1-Q4)...
    Q1 balance sheet done (8 calls)
    Q2 balance sheet done (16 calls)
    Q3 balance sheet done (24 calls)
    Q4 balance sheet done (32 calls)
  Fetching revenue data (Q1-Q4)...
  Fetching cash flow data...
  Fetching profitability data...
  Fetching public float data (Q1-Q4)...
  Total API calls: 57
  Fetching prior year revenue (Q1-Q4)...
  Fetching prior year gross profit (Q1-Q4)...
Step 2: Building company list...
  5825 companies on NYSE/Nasdaq
  4244 with financial data
Step 3: Scoring companies...

Done! Screened 4244 companies.
  High-risk companies found: 19


### High-Risk Companies (19 total, showing top 50)

,Ticker,Company,Score,Risk Level,Key Drivers
0,CBRL,"CRACKER BARREL OLD COUNTRY STORE, INC",61.0,HIGH,Low cash runway: 2.2 months; Weak liquidity: C...
1,DNUT,"Krispy Kreme, Inc.",59.0,MEDIUM,Low cash runway: 10.8 months; Weak liquidity: ...
2,FUN,Six Flags Entertainment Corporation/NEW,58.0,MEDIUM,Low cash runway: 4.1 months; Weak liquidity: C...
3,SMG,SCOTTS MIRACLE-GRO CO,55.0,MEDIUM,Low cash runway: 0.5 months; $55M debt due in ...
4,LGIH,"LGI Homes, Inc.",54.0,MEDIUM,Low cash runway: 0.0 months; Weak liquidity: C...
5,BALL,BALL Corp,53.5,MEDIUM,Low cash runway: 8.1 months; $447M debt due in...
6,CIFR,Cipher Digital Inc.,53.5,MEDIUM,Low cash runway: 5.9 months; $38M debt due in ...
7,PLCE,"Childrens Place, Inc.",52.5,MEDIUM,Low cash runway: 1.6 months; Weak liquidity: C...
8,KLXE,"KLX Energy Services Holdings, Inc.",52.0,MEDIUM,Low cash runway: 4.7 months; Negative operatin...
9,DY,DYCOM INDUSTRIES INC,51.0,MEDIUM,Low cash runway: 3.6 months; $15M debt due in ...


## Score Interpretation Guide

### Risk Levels
- **Critical** (75-100): Immediate capital raise very likely
- **High** (60-74): Strong likelihood of capital raise in next 12 months
- **Medium** (40-59): Moderate risk, monitor closely
- **Low** (0-39): No immediate capital raise pressure

### Threshold
- **Score > 50**: Company flagged for monitoring
- **Score ≤ 50**: Below concern threshold

### Key Signals
1. **Cash Runway** (0-40 pts): Most important. < 12 months is concerning.
2. **Liquidity Stress** (0-20 pts): Current ratio < 1.0 is critical.
3. **Debt Maturity** (0-15 pts): Large obligations due in 6-18 months.
4. **Operational Red Flags** (0-15 pts): Revenue decline, negative FCF.
5. **Market & Behavioral** (0-10 pts): Stock down, insider selling.
